# Forecasting 3 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Sportswear retail

**Problem:** Forecast next-week store sales for staffing and stock planning.

**Proposed success criterion:** Beat a four-week rolling-mean baseline MAE by at least 10% on the last 12 weeks; compare stores.

**Dataset:** `forecasting_3.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [3]:
pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 53.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_3.csv')
if not DATA.exists(): DATA = Path('datasets/forecasting_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(841, 12)


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur,list_price_eur
0,S01,01/01/2024,Yes,0,464,13.5,579,Large,14.81,242.75,13098.18,17.18
1,S01,08/01/2024,Yes,0,453,5.0,579,Large,15.25,262.27,14986.99,17.69
2,S01,15/01/2024,Yes,0,445,13.9,579,Large,14.48,485.51,13223.24,16.80
3,S01,22/01/2024,Yes,0,446,16.7,579,Large,14.73,398.76,15302.44,17.09
4,S01,29/01/2024,No,0,395,7.1,579,Large,13.73,180.64,13367.07,15.93


store_id                  0
week_start                0
promo_label               0
holiday_week              0
footfall_actual           0
rain_mm_actual            0
store_floor_m2            0
store_size_category       0
average_price_eur         0
returns_eur_after_week    0
sales_eur                 0
list_price_eur            0
dtype: int64

Exact duplicates: 1


This forecast supports the store manager and regional planner, who use it every week to decide staffing levels and how much stock to request for the coming week. The prediction moment is the end of the current week, before next week starts, so only information already known by then can be used as input. The observation unit is one store in one week, identified by store_id and week_start, and the target is sales_eur.

The proposed success criterion, beating a four week rolling mean baseline by 10 percent MAE, is measurable but incomplete on its own. A single overall MAE can hide the fact that some stores are harder to forecast than others, and MAE treats overestimating and underestimating demand as equally costly, which they are probably not for staffing and stock decisions. We propose keeping the MAE comparison as the main technical measure, but reporting it per store rather than only as one combined number, and separately tracking the share of weeks where demand is underestimated versus overestimated.

Two evidenced findings: the dataset is a complete, balanced weekly panel of 8 stores over 105 weeks each, with no missing values, so coverage is not a concern. We also found one exact duplicate row (store S02, week of 20 May 2024) and one implausible sales value of 82,000 euros for store S03, far above the next highest value of about 18,250 euros, both of which need to be addressed before modelling.

Two limitations: the returns_eur_after_week field is only known after the week is over, so it cannot be used as a predictor without leaking future information into the forecast. Also, promo_label contains four different spellings for what appears to be only two real categories, so the true promotional effect cannot be measured until these labels are cleaned.

## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [5]:
# Add your preparation code below. Inspect before/after evidence.
# Make a copy first:
work = df.copy()

work['week_start'] = pd.to_datetime(work['week_start'], dayfirst=True)

before = work.shape[0]
work = work.drop_duplicates()
print('Exact duplicates removed:', before - work.shape[0])

print('Before:', work['promo_label'].value_counts())
work['promo_flag'] = work['promo_label'].replace({'Yes': 1, 'promo': 1, 'No': 0, 'REGULAR': 0})
print('After:', work['promo_flag'].value_counts())

print('Correlation:', work[['average_price_eur', 'list_price_eur']].corr().iloc[0, 1])
work = work.drop(columns=['list_price_eur'])

work = work.drop(columns=['returns_eur_after_week'])

store_median = work.groupby('store_id')['sales_eur'].transform('median')
implausible = work['sales_eur'] > 3 * store_median
print('Implausible rows capped:', implausible.sum())
work.loc[implausible, 'sales_eur'] = store_median[implausible]

Exact duplicates removed: 1
Before: promo_label
No         638
Yes        196
REGULAR      4
promo        2
Name: count, dtype: int64
After: promo_flag
0    642
1    198
Name: count, dtype: int64
Correlation: 0.9999904317803365
Implausible rows capped: 1


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
|Exact duplicate row (S02, 2024-05-20)|Dropped the duplicate record|Row count 841 to 840|Counting it twice would double that store-week in training|Proceed with the deduplicated panel|
|promo_label had four inconsistent labels|Mapped to a clean binary promo_flag|638/196/4/2 before, clean 642/198 split after|Without this the promotion effect cannot be measured reliably|Use promo_flag as the predictor|
|average_price_eur and list_price_eur correlate at 0.9999|Dropped list_price_eur, kept average_price_eur|Correlation of 0.99999 before, single price field after|Keeping both adds no information and risks instability in linear models|Use average_price_eur only|

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

In [ ]:
# Build and test your baseline and candidate model here.
# Suggested tools: sklearn Pipeline, ColumnTransformer, SimpleImputer,
# OneHotEncoder, LogisticRegression, RandomForestRegressor or Ridge.
# For forecasting: use time-ordered validation; no future sales or actual future footfall.

## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.